# QM640 · 01 Data acquisition and profiling (standalone Colab version)

Pallab Bhattacharya · Walsh College QM640 capstone.
Loads the **UCI ServiceNow** and **Rabobank BPI 2014** datasets from your Drive folder, profiles every variable, and measures friction count and time to action. Results are saved to Drive.

**Before running:** put the Rabobank CSV files in `Walsh DBA/Doctoral Research/QM640/Data/raw/bpi2014/` (ServiceNow downloads automatically). Then *Runtime → Run all*.

Results and a step summary (`results/step01_summary.md`) are written to Drive so Claude can review them directly.

In [ ]:
# 1 · Mount Drive and set folders
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
DATA_ROOT = Path('/content/drive/MyDrive/Walsh DBA/Doctoral Research/QM640/Data')
RAW, PROCESSED, RESULTS = DATA_ROOT/'raw', DATA_ROOT/'processed', DATA_ROOT/'results'
for p in (RAW/'servicenow', RAW/'bpi2014', PROCESSED, RESULTS): p.mkdir(parents=True, exist_ok=True)
print('Data folder:', DATA_ROOT)
print('Rabobank files found:', [f.name for f in (RAW/'bpi2014').glob('*')])

In [ ]:
%%writefile /content/capstone_data.py
"""Data acquisition and preparation for the QM640 capstone.

Three real, public ITSM datasets are used:

* ServiceNow incident event log (UCI ML Repository, id 498, CC BY 4.0)
* Rabobank ITSM log, BPI Challenge 2014 (4TU.ResearchData)
* The Public Jira Dataset (Montgomery, Lueders & Maalej, 2022; Zenodo v7, CC BY 4.0)

Downloads need open internet access (run in Google Colab). Every function
that touches the network is separate from the functions that parse files,
so parsing can be tested offline.
"""
from __future__ import annotations

import io
import json
import re
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd

UCI_SERVICENOW_ZIP = (
    "https://archive.ics.uci.edu/static/public/498/"
    "incident+management+process+enriched+event+log.zip"
)
# 4TU.ResearchData article ids for the four BPI Challenge 2014 files
BPI2014_ARTICLES = {
    "incident": 12692378,
    "interaction": 12692411,
    "incident_activity": 12706424,
    "change": 12716234,
}
JIRA_ZENODO_RECORD = 15719919  # v7, 2025-06-23, anonymised, 5.8 GB zip


# ---------------------------------------------------------------------------
# helpers
# ---------------------------------------------------------------------------
def snake(name: str) -> str:
    """'CI Name (aff)' -> 'ci_name_aff', '# Reassignments' -> 'n_reassignments'."""
    name = name.strip().replace("#", "n ")
    name = re.sub(r"[^0-9a-zA-Z]+", "_", name)
    return re.sub(r"_+", "_", name).strip("_").lower()


def download(url: str, dest: Path, chunk: int = 1 << 20) -> Path:
    """Stream a URL to disk (skips if the file already exists)."""
    import requests

    dest = Path(dest)
    if dest.exists() and dest.stat().st_size > 0:
        return dest
    dest.parent.mkdir(parents=True, exist_ok=True)
    with requests.get(url, stream=True, timeout=120) as r:
        r.raise_for_status()
        tmp = dest.with_suffix(dest.suffix + ".part")
        with open(tmp, "wb") as fh:
            for block in r.iter_content(chunk):
                fh.write(block)
        tmp.rename(dest)
    return dest


# ---------------------------------------------------------------------------
# ServiceNow (UCI 498)
# ---------------------------------------------------------------------------
def fetch_servicenow(raw_dir: Path) -> Path:
    """Download and unzip the UCI ServiceNow event log; returns the CSV path."""
    raw_dir = Path(raw_dir) / "servicenow"
    z = download(UCI_SERVICENOW_ZIP, raw_dir / "servicenow.zip")
    with zipfile.ZipFile(z) as zf:
        zf.extractall(raw_dir)
    csvs = sorted(raw_dir.rglob("*.csv"))
    if not csvs:
        raise FileNotFoundError(f"No CSV found after unzipping {z}")
    return csvs[0]


SN_DATES = ["opened_at", "sys_created_at", "sys_updated_at", "resolved_at", "closed_at"]


def load_servicenow_events(csv_path: Path) -> pd.DataFrame:
    """Read the event log; '?' means unknown, as the source documentation states."""
    df = pd.read_csv(csv_path, na_values=["?"], low_memory=False)
    for c in SN_DATES:
        if c in df.columns:
            df[c] = pd.to_datetime(df[c], dayfirst=True, errors="coerce")
    return df


def servicenow_to_incidents(events: pd.DataFrame) -> pd.DataFrame:
    """One row per incident: the last recorded event, plus derived measures."""
    ev = events.sort_values(["number", "sys_mod_count", "sys_updated_at"])
    inc = ev.groupby("number", as_index=False).tail(1).copy()
    inc["resolution_hours"] = (inc["resolved_at"] - inc["opened_at"]).dt.total_seconds() / 3600
    # Forrester-style friction count: distinct teams that handled the ticket
    inc["friction_count"] = inc["reassignment_count"].fillna(0).astype(int) + 1
    inc["n_events"] = ev.groupby("number")["number"].transform("size").loc[inc.index]
    return inc.reset_index(drop=True)


# ---------------------------------------------------------------------------
# Rabobank, BPI Challenge 2014 (4TU)
# ---------------------------------------------------------------------------
def bpi2014_file_urls(article_id: int) -> list[dict]:
    """Ask the 4TU API for the download links of one dataset.

    4TU.ResearchData exposes a Figshare-compatible API. If this call fails,
    download the CSV manually from https://data.4tu.nl/collections/_/5065469/1
    and place it in data/raw/bpi2014/.
    """
    import requests

    for url in (
        f"https://data.4tu.nl/v2/articles/{article_id}/files",
        f"https://data.4tu.nl/v2/articles/{article_id}",
    ):
        try:
            r = requests.get(url, timeout=60)
            if r.ok:
                data = r.json()
                files = data if isinstance(data, list) else data.get("files", [])
                if files:
                    return files
        except (requests.RequestException, json.JSONDecodeError):
            continue
    return []


def fetch_bpi2014(raw_dir: Path) -> dict[str, Path]:
    raw_dir = Path(raw_dir) / "bpi2014"
    out: dict[str, Path] = {}
    for name, art in BPI2014_ARTICLES.items():
        existing = [p for p in raw_dir.glob("*") if name.replace("_", "") in p.name.lower().replace("_", "")]
        if existing:
            out[name] = existing[0]
            continue
        files = bpi2014_file_urls(art)
        if not files:
            print(f"[manual] could not resolve {name}; download it from 4TU into {raw_dir}")
            continue
        f = files[0]
        url = f.get("download_url") or f.get("url")
        fname = f.get("name", f"{name}.csv")
        out[name] = download(url, raw_dir / fname)
        if out[name].suffix == ".zip":
            with zipfile.ZipFile(out[name]) as zf:
                zf.extractall(raw_dir)
    return out


def load_bpi_csv(path: Path) -> pd.DataFrame:
    """BPI 2014 files are semicolon-separated with day-first timestamps."""
    with open(path, encoding="latin-1") as fh:
        header = fh.readline()
    sep = ";" if header.count(";") > header.count(",") else ","  # files differ
    df = pd.read_csv(path, sep=sep, decimal="," if sep == ";" else ".", low_memory=False, encoding="latin-1")
    df = df.loc[:, ~df.columns.astype(str).str.startswith("Unnamed")]  # trailing separators
    df.columns = [snake(c) for c in df.columns]
    for c in df.columns:
        if any(k in c for k in ("time", "date", "start", "end")) and pd.api.types.is_string_dtype(df[c]):
            parsed = pd.to_datetime(df[c], format="%d/%m/%Y %H:%M:%S", errors="coerce")
            if parsed.notna().mean() < 0.5:
                parsed = pd.to_datetime(df[c], format="%d-%m-%Y %H:%M", errors="coerce")
            if parsed.notna().mean() > 0.5:
                df[c] = parsed
    return df


def bpi_incident_measures(inc: pd.DataFrame) -> pd.DataFrame:
    """Friction count and time-to-action measures for Rabobank incidents."""
    inc = inc.copy()
    reas = [c for c in inc.columns if "reassign" in c]
    if reas:
        inc["friction_count"] = pd.to_numeric(inc[reas[0]], errors="coerce").fillna(0).astype(int) + 1
    open_c = next((c for c in inc.columns if c.startswith("open_time")), None)
    res_c = next((c for c in inc.columns if c.startswith("resolved_time")), None)
    if open_c and res_c:
        for c in (open_c, res_c):
            inc[c] = pd.to_datetime(inc[c], dayfirst=True, errors="coerce")
        inc["resolution_hours"] = (inc[res_c] - inc[open_c]).dt.total_seconds() / 3600
    return inc


# ---------------------------------------------------------------------------
# Public Jira Dataset (Zenodo v7)
# ---------------------------------------------------------------------------
def jira_zip_url() -> str:
    return (
        f"https://zenodo.org/records/{JIRA_ZENODO_RECORD}/files/"
        "2025-06-23%20ThePublicJiraDataset.zip?download=1"
    )


JIRA_FIELDS = {
    "key": "key",
    "fields.summary": "summary",
    "fields.description": "description",
    "fields.issuetype.name": "issuetype",
    "fields.priority.name": "priority",
    "fields.status.name": "status",
    "fields.resolution.name": "resolution",
    "fields.created": "created",
    "fields.resolutiondate": "resolutiondate",
    "fields.assignee": "assignee",
    "fields.project.key": "project",
}


def _get(doc: dict, dotted: str):
    cur = doc
    for part in dotted.split("."):
        if not isinstance(cur, dict):
            return None
        cur = cur.get(part)
    return cur


def flatten_jira_issue(doc: dict, repository: str) -> dict:
    """Flatten one Jira REST document into the columns used in the study."""
    row = {"repository": repository}
    for src, dst in JIRA_FIELDS.items():
        v = _get(doc, src)
        if dst == "assignee" and isinstance(v, dict):
            v = v.get("key") or v.get("name") or v.get("accountId")
        row[dst] = v
    f = doc.get("fields", {}) or {}
    row["components"] = [c.get("name") for c in (f.get("components") or []) if isinstance(c, dict)]
    row["labels"] = list(f.get("labels") or [])
    links = []
    for ln in f.get("issuelinks") or []:
        t = (ln.get("type") or {}).get("name")
        other = ln.get("outwardIssue") or ln.get("inwardIssue") or {}
        direction = "outward" if "outwardIssue" in ln else "inward"
        links.append({"type": t, "direction": direction, "key": other.get("key")})
    row["issuelinks"] = links
    row["n_comments"] = len(((f.get("comment") or {}).get("comments")) or [])
    hist = ((doc.get("changelog") or {}).get("histories")) or []
    assignee_changes = sum(
        1 for h in hist for it in (h.get("items") or []) if (it.get("field") or "").lower() == "assignee"
    )
    row["n_assignee_changes"] = assignee_changes
    row["friction_count"] = max(assignee_changes, 1)
    return row


def jira_frame(rows: list[dict]) -> pd.DataFrame:
    df = pd.DataFrame(rows)
    for c in ("created", "resolutiondate"):
        if c in df:
            df[c] = pd.to_datetime(df[c], errors="coerce", utc=True)
    df["is_duplicate_resolution"] = df["resolution"].fillna("").str.lower().eq("duplicate")
    df["has_duplicate_link"] = df["issuelinks"].apply(
        lambda ls: any((l.get("type") or "").lower() == "duplicate" for l in ls)
    )
    df["resolution_hours"] = (df["resolutiondate"] - df["created"]).dt.total_seconds() / 3600
    return df


# ---------------------------------------------------------------------------
# profiling (used for the synopsis screenshots and Appendix A)
# ---------------------------------------------------------------------------
def profile(df: pd.DataFrame, name: str) -> pd.DataFrame:
    """Column-level profile: type, missing share, distinct values, example."""
    rows = []
    for c in df.columns:
        s = df[c]
        try:
            nunique = s.nunique(dropna=True)
        except TypeError:  # list-valued columns
            nunique = s.astype(str).nunique()
        ex = s.dropna().iloc[0] if s.notna().any() else None
        rows.append({
            "dataset": name,
            "variable": c,
            "dtype": str(s.dtype),
            "missing_pct": round(100 * s.isna().mean(), 1),
            "distinct": int(nunique),
            "example": str(ex)[:60],
        })
    return pd.DataFrame(rows)


In [ ]:
# 2 · Imports
import sys; sys.path.insert(0, '/content')
import importlib, capstone_data as D; importlib.reload(D)
import pandas as pd, numpy as np, matplotlib.pyplot as plt
pd.set_option('display.max_columns', 40); pd.set_option('display.width', 200)

## 3 · ServiceNow incident event log (UCI 498) — downloads automatically

In [ ]:
sn_csv = D.fetch_servicenow(RAW)
sn_events = D.load_servicenow_events(sn_csv)
print('events:', sn_events.shape, '| incidents:', sn_events['number'].nunique(),
      '| dates:', sn_events['opened_at'].min(), 'to', sn_events['opened_at'].max())

📸 Screenshot: first rows of the raw event log

In [ ]:
sn_events.head(10)

In [ ]:
sn = D.servicenow_to_incidents(sn_events)
sn.to_parquet(PROCESSED/'servicenow_incidents.parquet')
print('incident-level rows:', len(sn))
sn[['number','incident_state','category','priority','assignment_group','reassignment_count','friction_count','knowledge','made_sla','resolution_hours']].head()

📸 Screenshot: variable profile (supports the data dictionary)

In [ ]:
sn_prof = D.profile(sn_events, 'ServiceNow events')
sn_prof.to_csv(RESULTS/'profile_servicenow.csv', index=False)
sn_prof

In [ ]:
wp_sn = pd.Series({
  'incidents': len(sn),
  'mean friction count (teams per incident)': sn['friction_count'].mean(),
  '% incidents handled by >1 team': 100*(sn['friction_count']>1).mean(),
  '% knowledge-base used': 100*sn['knowledge'].astype(str).str.lower().eq('true').mean(),
  '% SLA met': 100*sn['made_sla'].astype(str).str.lower().eq('true').mean(),
  'median resolution hours': sn['resolution_hours'].median(),
}).round(2)
wp_sn

## 4 · Rabobank ITSM log (BPI Challenge 2014) — from your Drive folder
Files are matched by name: *incident* (not activity), *interaction*, *activity*, *change*.

In [ ]:
import zipfile
for z in (RAW/'bpi2014').glob('*.zip'):
    with zipfile.ZipFile(z) as zf: zf.extractall(RAW/'bpi2014')
def find_bpi(folder):
    files = [f for f in folder.rglob('*.csv')]
    low = {f: f.name.lower() for f in files}
    pick = lambda cond: next((f for f, n in low.items() if cond(n)), None)
    found = {
      'incident':          pick(lambda n: 'incident' in n and 'activity' not in n),
      'interaction':       pick(lambda n: 'interaction' in n),
      'incident_activity': pick(lambda n: 'activity' in n),
      'change':            pick(lambda n: 'change' in n),
    }
    return {k: v for k, v in found.items() if v is not None}
bpi_files = find_bpi(RAW/'bpi2014')
print(bpi_files)
bpi = {k: D.load_bpi_csv(p) for k, p in bpi_files.items()}
for k, df in bpi.items(): print(f'{k:18s}', df.shape)

📸 Screenshot: first rows of the incident file

In [ ]:
bpi['incident'].head(10) if 'incident' in bpi else 'incident file missing'

In [ ]:
bpi_prof = pd.concat([D.profile(df, f'Rabobank {k}') for k, df in bpi.items()])
bpi_prof.to_csv(RESULTS/'profile_rabobank.csv', index=False)
bpi_prof

In [ ]:
inc = D.bpi_incident_measures(bpi['incident'])
inc.to_parquet(PROCESSED/'rabobank_incidents.parquet')
rel = [c for c in inc.columns if 'related_change' in c]
wp_bpi = pd.Series({
  'incidents': len(inc),
  'mean friction count (teams per incident)': inc['friction_count'].mean() if 'friction_count' in inc else np.nan,
  '% incidents handled by >1 team': 100*(inc['friction_count']>1).mean() if 'friction_count' in inc else np.nan,
  'median resolution hours': inc['resolution_hours'].median() if 'resolution_hours' in inc else np.nan,
  '% incidents linked to a change': 100*inc[rel[0]].notna().mean() if rel else np.nan,
}).round(2)
wp_bpi

## 5 · Combined workload profile
📸 Screenshot: table and chart

In [ ]:
wp = pd.concat({'ServiceNow': wp_sn, 'Rabobank': wp_bpi}, axis=1)
wp.to_csv(RESULTS/'workload_profile.csv'); wp

In [ ]:
fig, ax = plt.subplots(figsize=(7, 3.5))
for label, s in [('ServiceNow', sn['friction_count']), ('Rabobank', inc.get('friction_count', pd.Series(dtype=int)))]:
    if len(s): s.clip(upper=8).value_counts(normalize=True).sort_index().mul(100).plot(ax=ax, marker='o', label=label)
ax.set_xlabel('Teams that handled the ticket (8 = 8 or more)'); ax.set_ylabel('% of tickets')
ax.set_title('Friction count per ticket'); ax.legend(); plt.tight_layout()
plt.savefig(RESULTS/'friction_count.png', dpi=200); plt.show()

Done. Results are in `QM640/Data/results` on Drive. Jira (5.8 GB) is handled later in the December notebook; for the synopsis, screenshot its Zenodo page.

## 6 · Data dictionary (interim-report format)

In [ ]:
# ---- Data dictionary in the interim-report format ----
def rng(s):
    s = s.dropna()
    if len(s) == 0: return "all missing"
    if pd.api.types.is_bool_dtype(s): return "True / False"
    if pd.api.types.is_numeric_dtype(s): return f"{s.min():g} to {s.max():g}"
    if pd.api.types.is_datetime64_any_dtype(s): return f"{s.min():%Y-%m-%d} to {s.max():%Y-%m-%d}"
    vc = s.astype(str).value_counts()
    top = ", ".join(vc.index[:3])
    return f"{len(vc)} values (e.g., {top})" if len(vc) > 3 else ", ".join(vc.index)

SN_DEF = {
 'number': ('Incident identifier', 'Key; never missing', 'Unit of analysis = incident'),
 'incident_state': ('Workflow state at each event', 'Last event kept per incident', 'Event-to-incident reduction'),
 'category': ('Incident category (anonymised)', 'Rows with missing category excluded from D1 training', 'D1 target'),
 'subcategory': ('Incident subcategory (anonymised)', 'Kept as feature; missing = "unknown"', 'D1 feature'),
 'u_symptom': ('Reported symptom (anonymised)', 'Missing = "unknown" category', 'D1/D5 feature'),
 'impact': ('ITIL impact', 'None missing', 'D1 feature'),
 'urgency': ('ITIL urgency', 'None missing', 'D1 feature'),
 'priority': ('ITIL priority (from impact × urgency)', 'None missing', 'D1 target (priority)'),
 'contact_type': ('Channel the incident arrived by', 'None missing', 'Feature'),
 'location': ('Caller location (anonymised)', 'Missing = "unknown"', 'Feature'),
 'assignment_group': ('Support team assigned', 'Rows missing group excluded from D5', 'D5 target'),
 'assigned_to': ('Resolver assigned (anonymised)', 'Not used as target (person-level)', 'Privacy: not published beyond IDs'),
 'reassignment_count': ('Times the incident changed team', 'None missing', 'Friction count = value + 1'),
 'reopen_count': ('Times reopened', 'None missing', 'Quality signal'),
 'knowledge': ('Knowledge-base article used', 'None missing', 'D3 ground truth'),
 'u_priority_confirmation': ('Priority confirmed by a person', 'None missing', 'Human-confirmation signal (RQ4)'),
 'made_sla': ('SLA target met', 'None missing', 'Outcome'),
 'opened_at': ('Time the incident was opened', 'None missing', 'Time split; RQ4 windows'),
 'resolved_at': ('Time the incident was resolved', 'Missing -> resolution time not computed', 'Outcome'),
 'closed_code': ('Closure code', 'Missing = "unknown"', 'D4 proxy'),
 'cmdb_ci': ('Configuration item', '99.7% missing -> dropped', 'Not usable for graph'),
 'caused_by': ('Change that caused the incident', '100% missing -> dropped', 'D6 not testable on ServiceNow'),
 'problem_id': ('Linked problem record', '98.4% missing -> kept as flag', 'Graph link'),
 'rfc': ('Linked request for change', '99.3% missing -> kept as flag', 'Graph link'),
}
RB_DEF = {
 'incident_id': ('Incident identifier', 'Key; never missing', 'Unit of analysis = incident'),
 'ci_name_aff': ('Configuration item affected', 'None missing', 'Graph node; D5/D6 feature'),
 'ci_type_aff': ('Type of configuration item', 'None missing', 'Feature'),
 'service_component_wbs_aff': ('Service component affected', 'None missing', 'Graph; D5 feature'),
 'impact': ('ITIL impact (1 high to 5 low)', 'Coerced to numeric', 'D1 feature'),
 'urgency': ('ITIL urgency', 'Coerced to numeric', 'D1 feature'),
 'priority': ('ITIL priority', 'Coerced to numeric', 'D1 target (priority)'),
 'category': ('Incident or request for information', 'None missing', 'D1 target'),
 'km_number': ('Knowledge article used', 'Missing = none used', 'D3 ground truth'),
 'n_reassignments': ('Times the incident changed team', 'Missing -> 0', 'Friction count = value + 1'),
 'open_time': ('Time opened', 'Parsed day-first; failures dropped', 'Time split'),
 'resolved_time': ('Time resolved', 'Missing -> resolution time not computed', 'Outcome'),
 'closure_code': ('How the incident was closed', 'Missing = "unknown"', 'D4 proxy'),
 'related_interaction': ('Service-desk contact linked', 'Missing = none', 'Graph link'),
 'related_change': ('Change recorded as related', 'Missing = none (98.8%)', 'D6 ground truth'),
 'handle_time_hours': ('Total handling time', 'Decimal comma converted', 'Cost proxy'),
}
def dictionary(df, defs, name):
    rows = []
    for v, (d, h, n) in defs.items():
        if v not in df: continue
        s = df[v]
        rows.append({'Dataset': name, 'Variable Name': v, 'Definition': d, 'Datatype': str(s.dtype),
                     'Allowed Values/Range': rng(s), 'Missing %': round(100*s.isna().mean(), 1),
                     'Missing Values Handling': h, 'Notes': n})
    return pd.DataFrame(rows)
dd = pd.concat([dictionary(sn, SN_DEF, 'ServiceNow (incident level)'), dictionary(inc, RB_DEF, 'Rabobank incidents')])
dd.to_csv(RESULTS/'data_dictionary.csv', index=False)
dd.head(12)


## 7 · Data cleaning log (counts and %), then apply treatments

In [ ]:
# ---- Data cleaning log (counts and %) ----
log = []
def add(issue, ds, var, detect, n, base, treat, why):
    log.append({'Issue': issue, 'Dataset': ds, 'Variables Affected': var, 'Detection Method': detect,
                'Count': int(n), '% of records': round(100*n/base, 2) if base else None,
                'Treatment Applied': treat, 'Rationale': why})
nE = len(sn_events)
add('Duplicates', 'ServiceNow', 'all columns', 'Exact duplicate event rows', sn_events.duplicated().sum(), nE, 'Dropped', 'Repeated audit rows add no information')
cells = sn_events.size
add('Missing values', 'ServiceNow', 'all 36 columns ("?" codes)', 'Cells coded "?" (unknown) in the source', sn_events.isna().sum().sum(), cells, 'Read as missing; per-variable rule in data dictionary; caused_by (100%) and cmdb_ci (99.7%) dropped', 'Follows dataset documentation; % is of all cells')
nI = len(sn)
add('Missing values', 'ServiceNow', 'resolved_at', 'Null after reduction', sn['resolved_at'].isna().sum(), nI, 'Resolution time left missing; incident kept', 'Incident still valid for D1/D5')
add('Missing values', 'ServiceNow', 'assignment_group', 'Null after reduction', sn['assignment_group'].isna().sum(), nI, 'Excluded from D5 only', 'No ground truth for assignment')
neg = (sn['resolution_hours'] < 0).sum()
add('Impossible values', 'ServiceNow', 'resolved_at < opened_at', 'Negative duration', neg, nI, 'Resolution time set to missing', 'Timestamp error')
q1, q3 = sn['resolution_hours'].quantile([.25, .75]); hi = q3 + 3*(q3-q1)
add('Outliers', 'ServiceNow', 'resolution_hours', f'IQR rule (> Q3 + 3·IQR = {hi:.0f} h)', (sn['resolution_hours'] > hi).sum(), nI, 'Kept; flagged; medians reported', 'Long-running incidents are real, not errors')
nR = len(inc)
add('Duplicates', 'Rabobank', 'incident_id', 'Duplicate keys', inc['incident_id'].duplicated().sum(), nR, 'Kept first record', 'One row per incident')
add('Format errors', 'Rabobank', 'Detail_Incident.csv', 'Mixed date formats and comma separator after re-saving in Excel', nR, nR, 'File re-downloaded from source; never opened in Excel', 'Excel swapped day/month and dropped fields')
add('Format errors', 'Rabobank', 'handle_time_hours', 'Decimal comma (European format)', inc['handle_time_hours'].notna().sum(), nR, 'Parsed with decimal=","', 'Correct numeric values')
add('Missing values', 'Rabobank', 'open_time / resolved_time', 'Null or unparseable', (inc['open_time'].isna() | inc['resolved_time'].isna()).sum(), nR, 'Resolution time not computed', 'Incident still valid for D1/D5')
add('Missing values', 'Rabobank', 'closure_code', 'Null', inc['closure_code'].isna().sum(), nR, 'Set to "unknown"; excluded from D4', 'No ground truth')
neg = (inc['resolution_hours'] < 0).sum()
add('Impossible values', 'Rabobank', 'resolved < opened', 'Negative duration', neg, nR, 'Resolution time set to missing', 'Timestamp error')
q1, q3 = inc['resolution_hours'].quantile([.25, .75]); hi = q3 + 3*(q3-q1)
add('Outliers', 'Rabobank', 'resolution_hours', f'IQR rule (> Q3 + 3·IQR = {hi:.0f} h)', (inc['resolution_hours'] > hi).sum(), nR, 'Kept; flagged; medians reported', 'Long-running incidents are real')
clog = pd.DataFrame(log); clog.to_csv(RESULTS/'cleaning_log.csv', index=False)
# apply treatments
sn_events = sn_events.drop_duplicates()
sn.loc[sn['resolution_hours'] < 0, 'resolution_hours'] = np.nan
inc = inc.drop_duplicates('incident_id'); inc.loc[inc['resolution_hours'] < 0, 'resolution_hours'] = np.nan
sn.to_parquet(PROCESSED/'servicenow_incidents.parquet'); inc.to_parquet(PROCESSED/'rabobank_incidents.parquet')
clog


## 8 · EDA figures and insight table (saved to results/figures)

In [ ]:
# ---- EDA figures with insights ----
FIG = RESULTS/'figures'; FIG.mkdir(exist_ok=True)
plt.rcParams.update({'font.size': 10, 'axes.spines.top': False, 'axes.spines.right': False})
ins = []
# Fig A: monthly volume
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
m1 = sn.set_index('opened_at').resample('MS')['number'].count()
m2 = inc.dropna(subset=['open_time']).set_index('open_time').resample('MS')['incident_id'].count()
m1.plot(ax=ax[0], marker='o', color='#2b6cb0'); ax[0].set_title('ServiceNow incidents per month'); ax[0].set_xlabel('')
m2[m2.index >= '2013-01-01'].plot(ax=ax[1], marker='o', color='#c05621'); ax[1].set_title('Rabobank incidents per month (2013+)'); ax[1].set_xlabel('')
plt.tight_layout(); plt.savefig(FIG/'eda_monthly_volume.png', dpi=200); plt.close()
sn_months = (m1 > 100).sum(); rb_months = (m2 > 100).sum()
ins.append(['eda_monthly_volume', 'Incidents opened per month', f'ServiceNow has {sn_months} months with >100 incidents; Rabobank has {rb_months}', 'RQ4 needs 25 monthly windows; neither log alone gives 25 dense months', 'Use Jira for the 25-month replay; logs for workload profile'])
# Fig B: category and priority imbalance
fig, ax = plt.subplots(1, 2, figsize=(10, 3.2))
vc = sn['category'].value_counts(); (vc.head(15)/len(sn)*100).plot.bar(ax=ax[0], color='#2b6cb0'); ax[0].set_title('ServiceNow: top 15 categories (% of incidents)'); ax[0].set_xlabel('')
(sn['priority'].value_counts(normalize=True).sort_index()*100).plot.bar(ax=ax[1], color='#2b6cb0'); ax[1].set_title('ServiceNow: priority (% of incidents)'); ax[1].set_xlabel('')
plt.tight_layout(); plt.savefig(FIG/'eda_class_imbalance.png', dpi=200); plt.close()
top5 = vc.head(5).sum()/len(sn)*100; modp = sn['priority'].value_counts(normalize=True).max()*100
ins.append(['eda_class_imbalance', 'Category and priority distribution', f'{vc.size} categories; top 5 = {top5:.0f}% of incidents; one priority level = {modp:.0f}%', 'Strong class imbalance: accuracy alone would flatter a model that predicts the majority class (RQ1)', 'Report macro-F1; stratified test sample'])
# Fig C: SLA vs friction
g = sn.assign(teams=sn['friction_count'].clip(upper=6)).groupby('teams').agg(n=('number','size'), sla=('made_sla', lambda s: 100*s.astype(bool).mean()), res=('resolution_hours','median'))
fig, ax = plt.subplots(figsize=(6.5, 3.4)); ax2 = ax.twinx()
ax.bar(g.index, g['sla'], color='#2b6cb0', alpha=.8, label='% SLA met'); ax.set_ylabel('% SLA met'); ax.set_xlabel('Teams that handled the incident (6 = 6 or more)')
ax2.plot(g.index, g['res'], color='#c05621', marker='o', label='Median resolution (h)'); ax2.set_ylabel('Median resolution (hours)')
ax.set_title('ServiceNow: SLA and resolution time by number of teams'); fig.legend(loc='upper right', bbox_to_anchor=(.88, .88), fontsize=8)
plt.tight_layout(); plt.savefig(FIG/'eda_sla_by_friction.png', dpi=200); plt.close()
g.to_csv(RESULTS/'sla_by_friction.csv')
s1, s3 = g.loc[1, 'sla'], g.loc[3:, 'sla'].mean() if len(g.loc[3:]) else np.nan
r1, r3 = g.loc[1, 'res'], g.loc[3, 'res'] if 3 in g.index else np.nan
ins.append(['eda_sla_by_friction', 'SLA and resolution vs teams involved', f'SLA met {s1:.0f}% with one team vs {s3:.0f}% with 3+ teams; median resolution {r1:.0f} h vs {r3:.0f} h', 'Each misrouting hand-off costs time and SLA: an accurate first assignment (D5) has direct business value (RQ1–RQ2)', 'Weight D5 errors by hand-off cost in the cost model'])
# Fig D: resolution by priority (log scale)
fig, ax = plt.subplots(figsize=(6.5, 3.2))
pr = sorted(sn['priority'].dropna().unique())
ax.boxplot([sn.loc[sn['priority']==p, 'resolution_hours'].dropna().clip(lower=.1) for p in pr], tick_labels=pr, showfliers=False)
ax.set_yscale('log'); ax.set_ylabel('Resolution hours (log)'); ax.set_title('ServiceNow: resolution time by priority')
plt.tight_layout(); plt.savefig(FIG/'eda_resolution_by_priority.png', dpi=200); plt.close()
med = sn.groupby('priority')['resolution_hours'].median()
ins.append(['eda_resolution_by_priority', 'Resolution time by priority', '; '.join(f'{k}: {v:.0f} h' for k, v in med.items()), 'Priority is a risk feature for the allocation tree (error cost differs by priority)', 'Include priority as a decision-risk feature'])
# Fig E: knowledge use
k = sn.groupby(sn['knowledge'].astype(bool)).agg(n=('number','size'), res=('resolution_hours','median'), sla=('made_sla', lambda s: 100*s.astype(bool).mean()))
fig, ax = plt.subplots(figsize=(5.5, 3)); ax.bar(['No knowledge article', 'Knowledge article used'], k['res'], color=['#a0aec0', '#2b6cb0'])
ax.set_ylabel('Median resolution (hours)'); ax.set_title('ServiceNow: knowledge use and resolution time')
plt.tight_layout(); plt.savefig(FIG/'eda_knowledge.png', dpi=200); plt.close()
ins.append(['eda_knowledge', 'Resolution time with and without a knowledge article', f'Knowledge used in {100*k.loc[True,"n"]/len(sn):.0f}% of incidents; median {k.loc[True,"res"]:.0f} h with vs {k.loc[False,"res"]:.0f} h without', 'Knowledge articles are used mostly on harder, longer incidents (selection effect, not a causal slowdown); RQ3 must compare like with like', 'RQ3 compares instruments on the same tickets with and without context'])
# Rabobank first-call resolution
fcr = bpi['interaction']['first_call_resolution'].astype(str).str.upper().eq('Y').mean()*100
ins.append(['workload_profile', 'Workload profile (Table)', f'Rabobank: {fcr:.0f}% of service-desk contacts resolved at first call; {100*(inc["friction_count"]>1).mean():.0f}% of incidents need >1 team', 'Shows decision volumes the instruments would serve', 'Weight costs by real volumes'])
eda = pd.DataFrame(ins, columns=['Figure/Table Reference', 'What it Shows', 'Key Insight', 'Why it Matters (Link to RQ/objective)', 'Decision/Next Step'])
eda.to_csv(RESULTS/'eda_insights.csv', index=False)
eda


## 9 · Write step summary to Drive (for Claude's review)

In [ ]:
import datetime, io
def mdt(df, **kw):
    try: return df.to_markdown(**kw)
    except Exception: return '```\n' + df.to_string() + '\n```'
def g(name):
    return globals().get(name)
lines = [f"# Step 01 summary — {datetime.datetime.now():%Y-%m-%d %H:%M}", ""]
sn_ev, snI = g('sn_events'), g('sn')
if sn_ev is not None:
    lines += ["## ServiceNow", f"- events: {len(sn_ev):,}; incidents: {sn_ev['number'].nunique():,}; columns: {sn_ev.shape[1]}",
              f"- opened_at range: {sn_ev['opened_at'].min()} to {sn_ev['opened_at'].max()}", ""]
else: lines += ["## ServiceNow", "- NOT LOADED", ""]
bp = g('bpi') or {}
lines += ["## Rabobank BPI 2014", f"- files found: {[str(p.name) for p in (g('bpi_files') or {}).values()]}"]
for k, df in bp.items():
    lines.append(f"- {k}: {df.shape[0]:,} rows x {df.shape[1]} cols; columns: {list(df.columns)}")
lines.append("")
wp = g('wp')
if wp is not None:
    lines += ["## Workload profile", mdt(wp), ""]
for nm in ['sn_prof', 'bpi_prof']:
    pr = g(nm)
    if pr is not None:
        lines += [f"## {nm}", mdt(pr, index=False), ""]
for nm, title in [('clog','Cleaning log'),('eda','EDA insights')]:
    t = g(nm)
    if t is not None: lines += [f'## {title}', mdt(t, index=False), '']
lines += ["## Python / package versions", f"- pandas {pd.__version__}, numpy {np.__version__}"]
(RESULTS/'step01_summary.md').write_text("\n".join(lines))
print("\n".join(lines[:25])); print('... written to', RESULTS/'step01_summary.md')